# ESOREX quickstart: predicting an enzyme's substrate preferences

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UCB-BioE-Anderson-Lab/ESOREX/blob/main/notebooks/tyrb_quickstart.ipynb)

Enzymes accept some molecules and reject others. **From only a handful of measured substrates,**
**can we predict an enzyme's activity on molecules it has never seen, and know which predictions**
**to trust?**

ESOREX trains on the 9 natural amino-acid substrates of *E. coli* **TyrB** (an aromatic-preferring
aminotransferase) and predicts 14 *unnatural* analogs. The last cell is the payoff: it ranks the
unseen substrates well **and** its own uncertainty score turns out to predict exactly where it is
wrong. Runs in about a minute.

## 1. Install and fetch the data

Only RDKit needs installing (NumPy, SciPy, pandas ship with Colab). We clone the repo for the
`esorex` package and the TyrB dataset.

In [ ]:
!pip install -q rdkit          # numpy, scipy and pandas are preinstalled in Colab (do not downgrade them)
!git clone -q https://github.com/UCB-BioE-Anderson-Lab/ESOREX.git
import sys
sys.path.insert(0, "/content/ESOREX")
DATA = "/content/ESOREX/data/transaminases/ONUFFER_curated.csv"

## 2. Give the model only the 9 naturals

TyrB prefers aromatic side chains. We hand it just the 9 amino acids it evolved with; every other
measured substrate is held out as an unseen test. The **reactive core** (amine, alpha-carbon,
carboxyl) is matched with SMARTS; specificity is learned from everything outside it.

In [ ]:
import csv
from rdkit import Chem

RATE_COL  = "eTATase_kf_over_KD_M-1_s-1"
DUPLICATE = {"Arginine (mu = 1.0)"}
NATURALS  = {"Aspartate", "Glutamate", "Phenylalanine", "Tryptophan", "Tyrosine",
             "Alanine", "Leucine", "Valine", "Arginine (mu = 0.2)"}
AA_CORE   = Chem.MolFromSmarts("[NX3][CX4H][CX3](=O)[OX2]")

def core(mol):
    return set(mol.GetSubstructMatch(AA_CORE))

def parse_rate(s):
    s = s.strip()
    if not s or s.lower() in ("nd", "n/a", "na", "-"): return None
    try: return float(s)
    except ValueError: return None

naturals, analogs = [], []
with open(DATA) as f:
    for row in csv.DictReader(f):
        name = row["substrate"].strip()
        if name in DUPLICATE: continue
        rate = parse_rate(row[RATE_COL])
        mol  = Chem.MolFromSmiles(row.get("substrate_smiles", "").strip())
        if rate is None or mol is None or not core(mol): continue
        (naturals if name in NATURALS else analogs).append((name, mol, rate))

print(f"trained on {len(naturals)} naturals; predicting {len(analogs)} unseen analogs")

## 3. Train, reproducing every measurement exactly

ESOREX converts rates to activation energies (`E = -RT ln k`) and solves the free-energy
decomposition as a **hard constraint**. `exact fit: True` means it reproduces all 9 training
rates exactly, interpolation, not a lossy regression.

In [ ]:
from esorex.energetic_specificity import EnergeticSpecificityModel

model = EnergeticSpecificityModel()
info = model.train([m for _, m, _ in naturals],
                   [core(m) for _, m, _ in naturals],
                   rates=[r for _, _, r in naturals])
print("exact fit:", info["exact"])

## 4. Predict the 14 unseen analogs, with a novelty score

Every prediction carries a **novelty** score: `0` means the value is pinned by the training data;
higher means the model is reaching into chemistry the 9 naturals never covered. This is the
model's own estimate of how far it is extrapolating, before we know the answer.

In [ ]:
import pandas as pd
from scipy.stats import spearmanr

rows = []
for name, mol, measured in analogs:
    p = model.predict(mol, core(mol))
    rows.append(dict(substrate=name, measured_rate=measured,
                     predicted_rate=round(p.rate, 1), novelty=round(p.novelty, 3)))

df = pd.DataFrame(rows).sort_values("novelty").reset_index(drop=True)
rho = spearmanr(df.measured_rate, df.predicted_rate).correlation
print(f"held-out rank correlation (Spearman rho) = {rho:.2f}")
df   # sorted low-novelty (confident) first

## 5. The payoff: the model knows where it will fail

Predicted vs measured for the 14 unseen analogs, each point colored by the model's **own novelty**
**score**. Points on the dashed line are accurate. The striking part: the **hot (high-novelty)**
points, the ones the model flags before seeing the answer, are exactly the ones that fall off the
line. Its uncertainty is calibrated.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

logerr = (np.log10(df.predicted_rate) - np.log10(df.measured_rate)).abs()
fig, ax = plt.subplots(figsize=(6.8, 5.6))
sc = ax.scatter(df.measured_rate, df.predicted_rate, c=df.novelty, cmap="YlOrRd",
                vmin=0, s=85, edgecolor='#333', linewidth=0.6, zorder=3)
lims = [min(df.measured_rate.min(), df.predicted_rate.min()) * 0.3,
        max(df.measured_rate.max(), df.predicted_rate.max()) * 3]
ax.plot(lims, lims, "--", color="#9aa0a6", zorder=1, label="perfect prediction")
(ax.set_xscale("log"), ax.set_yscale("log"))
ax.set_xlim(lims); ax.set_ylim(lims)
(ax.set_xlabel("measured rate  (kf/KD, M$^{-1}$s$^{-1}$)"), ax.set_ylabel("predicted rate"))
cb = fig.colorbar(sc); cb.set_label("novelty  (0 = pinned by data,  high = extrapolating)")
nov_err = spearmanr(df.novelty, logerr).correlation
ax.set_title(f"TyrB: 9 trained, {len(df)} predicted unseen\n"
             f"rank accuracy rho = {rho:.2f}   |   novelty-vs-error rho = {nov_err:.2f}", fontsize=11)
w = df.loc[logerr.idxmax()]
ax.annotate(f"{w.substrate}: flagged most novel,\nand the single worst miss (~{10**logerr.max():.0f}x off)",
            xy=(w.measured_rate, w.predicted_rate), xycoords='data',
            xytext=(0.03, 0.26), textcoords='axes fraction', fontsize=9, color='#8a2b0e',
            arrowprops=dict(arrowstyle='->', color='#8a2b0e', lw=0.9))
ax.legend(loc="upper left", fontsize=9, framealpha=0.95)
plt.tight_layout(); plt.show()

### The point

From just **9 measurements**, ESOREX ranks 14 substrates it never saw (Spearman rho about 0.73).
But the sharper result is **calibration**: the model's novelty score tracks its actual error
(Spearman about 0.66). Its worst prediction, **2-aminooctanoate** (~900x off, a long aliphatic
chain with no aromatic ring, a combination absent from the naturals), is the very substrate it
flags as most novel. Everything it is confident about lands within a few-fold.

**ESOREX predicts, and tells you which predictions to trust.** That is what a black-box regressor
cannot do.

Next: the same model does regioselectivity (FucTIII) in `docs/demonstrations/`; the concept pages
are in `docs/`. Point ESOREX at your own atom-mapped reactions and rates to model a different enzyme.